# Eurozone inflation and unemployment

This notebook audits and estimates the relationship between Eurostat''s **published annual HICP rate** and unemployment in eight euro-area economies from 2000 to 2025. The HICP series is already an annual rate of change, so it is never transformed with `pct_change(12)`.


In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import SVG, display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from eurozone_inflation.config import load_config
from eurozone_inflation.data import load_snapshot
from eurozone_inflation.analysis import descriptive_summary, fit_models, prepare_model_data


## 1. Load the validated panel

The committed snapshot makes the published result reproducible even when Eurostat revises live data. The loader asserts expected columns, complete coverage, and one row per country-month.


In [2]:
config = load_config(ROOT / "config" / "analysis.toml")
panel, quality = load_snapshot(
    ROOT / "data" / "processed" / "eurozone_macro_panel.csv", config
)
pd.Series(quality, name="value").to_frame()


                          value
countries                     8
months                      312
expected_rows              2496
complete_rows              2496
missing_hicp                  0
missing_unemployment          0
duplicate_country_months      0
start                   2000-01
end                     2025-12

## 2. Inspect the outcome

`hicp_annual_rate` is measured in percentage points and comes directly from `prc_hicp_manr` (`RCH_A`, `CP00`). The trend highlights the exceptional common inflation shock in 2021–2023.


In [3]:
monthly, descriptive = descriptive_summary(panel)
pd.Series(descriptive, name="value").to_frame()


                                value
peak_month                    2022-10
peak_mean_inflation           11.3375
mean_inflation_2010_2019       1.4560
mean_inflation_2021_2023       5.4639
mean_inflation_2025            2.4146

In [4]:
display(SVG(filename=str(ROOT / "figures" / "inflation_trend.svg")))


<IPython.core.display.Image object>

## 3. Estimate two fixed-effects specifications

Unemployment is lagged six months within each country. Model 1 removes time-invariant country differences. Model 2 also absorbs common calendar-month shocks. Both use standard errors clustered by country.


In [5]:
model_data = prepare_model_data(panel, config.unemployment_lag_months)
results, fitted = fit_models(model_data)
results.round(4)


                              model  coefficient  std_error  p_value  observations
0             Country fixed effects      -0.2586     0.0548   0.0022          2448
1  Country + month fixed effects      -0.1064     0.0130   0.0001          2448

In [6]:
display(SVG(filename=str(ROOT / "figures" / "model_coefficients.svg")))


<IPython.core.display.Image object>

## 4. Interpretation

Month effects attenuate the unemployment coefficient, showing that common shocks explain a meaningful share of inflation variation. The corrected estimate remains negative; it does not reverse sign. The coefficient is an association, not a causal effect, and inference with eight country clusters should be treated cautiously.


In [7]:
display(SVG(filename=str(ROOT / "figures" / "phillips_partial_scatter.svg")))


<IPython.core.display.Image object>

## 5. Reproducibility

The complete pipeline is available in `src/eurozone_inflation/`. Run `python scripts/run_analysis.py` to rebuild all tables and figures from the snapshot, or add `--refresh` to request a new Eurostat snapshot with explicit filters. Automated tests verify the measurement definition, panel uniqueness, model sample, coefficients, notebook execution, and nonempty figures.
